In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import selfies as sf
from rdkit import Chem
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



In [ ]:
DATA_CSV = "../smiles_selfies_full.csv"
CKPT_PATH = "trained_models/final(H256-L512-linpool).pt"
OUTPUT_DIR = "../artifacts/model_validation"



In [ ]:
RUN_PHASE2 = False
REUSE_STEP3_LATENTS_FOR_PHASE3 = True
RUN_PHASE5_SEMANTIC_INTERP = True


In [ ]:
df = pd.read_csv(DATA_CSV)
required_cols = {"smiles", "selfies"}
assert required_cols.issubset(df.columns), f"CSV must contain columns: {required_cols}"

df["tokens"] = df["selfies"].apply(lambda x: list(sf.split_selfies(x)))

all_tokens = [tok for seq in df["tokens"] for tok in seq]
PAD = "<PAD>"
SOS = "<SOS>"
EOS = "<EOS>"
vocab = [PAD, SOS, EOS] + sorted(set(all_tokens))
vocab_size = len(vocab)

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for tok, idx in tok2id.items()}

PAD_ID = 0
SOS_ID = 1
EOS_ID = 2



In [ ]:
def full_molecule_tokens_to_ids(tokens, tok2id):
    return np.array([SOS_ID] + [tok2id[t] for t in tokens] + [EOS_ID], dtype=np.int64)



def decode_tokens_to_selfies(token_ids, id2tok, pad_id=0, sos_id=1, eos_id=2):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()

    tokens = []
    for t in token_ids:
        t = int(t)

        if t == eos_id:
            break
        if t in (pad_id, sos_id):
            continue

        tok = id2tok.get(t)
        if tok is None:
            continue

        if tok == EOS:
            break
        if tok in (PAD, SOS):
            continue

        tokens.append(tok)

    return "".join(tokens)


def selfies_to_smiles(selfies_str):
    if selfies_str is None:
        return None
    if not isinstance(selfies_str, str):
        selfies_str = str(selfies_str)
    if selfies_str == "":
        return None
    try:
        return sf.decoder(selfies_str)
    except Exception:
        return None


def smiles_to_mol(smiles_str):
    if smiles_str is None:
        return None
    if not isinstance(smiles_str, str):
        smiles_str = str(smiles_str)
    if smiles_str == "":
        return None
    try:
        return Chem.MolFromSmiles(smiles_str)
    except Exception:
        return None


def canon_smiles(smiles_str):
    mol = smiles_to_mol(smiles_str)
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None


df["token_ids"] = df["tokens"].apply(lambda toks: full_molecule_tokens_to_ids(toks, tok2id))
df["lengths"] = df["token_ids"].apply(len)

sequences = df["token_ids"].tolist()
max_len = max(len(seq) for seq in sequences)
padded_data = np.zeros((len(sequences), max_len), dtype=np.int64)

for i, seq in enumerate(sequences):
    padded_data[i, : len(seq)] = seq

data = padded_data
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, shuffle=True)



In [ ]:
from pathlib import Path
import importlib.util

MODEL_PY = Path("models/final_nat_linearpool.py")
if not MODEL_PY.exists():
    MODEL_PY = Path("transformer_vae/models/final_nat_linearpool.py")
assert MODEL_PY.exists(), f"Model file not found: {MODEL_PY}"

spec = importlib.util.spec_from_file_location("final_nat_linearpool", MODEL_PY)
final_nat_linearpool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(final_nat_linearpool)

VaeTransformer = final_nat_linearpool.VaeTransformer
vae_loss = final_nat_linearpool.vae_loss
load_final_model = final_nat_linearpool.load_final_model

FINAL_HIDDEN_SIZE = final_nat_linearpool.FINAL_HIDDEN_SIZE
FINAL_ATTENTION_HEADS = final_nat_linearpool.FINAL_ATTENTION_HEADS
FINAL_NUM_SLOTS = final_nat_linearpool.FINAL_NUM_SLOTS
FINAL_LAYERS = final_nat_linearpool.FINAL_LAYERS
FINAL_LATENT_SIZE = final_nat_linearpool.FINAL_LATENT_SIZE



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

hidden_size = FINAL_HIDDEN_SIZE
attention_heads = FINAL_ATTENTION_HEADS
num_slots = FINAL_NUM_SLOTS
layers = FINAL_LAYERS
latent_size = FINAL_LATENT_SIZE

model = load_final_model(
    vocab_size=vocab_size,
    max_len=max_len,
    ckpt_path=CKPT_PATH,
    device=device,
)

print("Model loaded")
print(f"vocab_size={vocab_size}")
print(f"max_len={max_len}")
print(f"device={device}")



## Phase 1: Reconstruction Metrics (train / val / test)
- token accuracy
- exact sequence accuracy
- length metrics (MAE/RMSE + plots)
- reconstruction validity (SELFIES -> SMILES -> RDKit sanitize)
- summary table per split + CSV + metrics dict entries


In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader

PHASE1_BATCH_SIZE = 1024
PHASE1_DIR = Path(OUTPUT_DIR)
PHASE1_DIR.mkdir(parents=True, exist_ok=True)

if "metrics" not in globals() or not isinstance(metrics, dict):
    metrics = {}


def _ids_to_selfies(pred_ids):
    tokens = [
        id2tok[int(t)]
        for t in pred_ids
        if int(t) not in (PAD_ID, SOS_ID, EOS_ID)
    ]
    return "".join(tokens)


def _reconstruction_is_valid(selfies_str):
    try:
        smiles = sf.decoder(selfies_str)
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        return mol is not None
    except Exception:
        return False


@torch.no_grad()
def evaluate_phase1_split(split_name, split_array, batch_size=PHASE1_BATCH_SIZE):
    model.eval()
    loader = DataLoader(split_array, batch_size=batch_size, shuffle=False)

    total_tok = 0
    total_correct = 0
    total_seq = 0
    perfect = 0

    true_len_rows = []
    pred_len_rows = []
    length_error_rows = []
    exact_match_rows = []
    valid_smiles_rows = []

    for x in tqdm(loader, desc=f"Phase1 {split_name}"):
        x = x.to(device)
        logits, _, _, pred_len = model(x, mode="eval")
        preds = logits.argmax(dim=-1)
        pred_len_i = torch.round(pred_len).long().clamp(min=1)

        for i in range(x.size(0)):
            true = x[i]
            true_len = int((true != PAD_ID).sum().item())
            if true_len == 0:
                continue

            pred_acc = preds[i]
            if pred_acc.numel() < true_len:
                pad = torch.full(
                    (true_len - pred_acc.numel(),),
                    PAD_ID,
                    device=pred_acc.device,
                    dtype=pred_acc.dtype,
                )
                pred_acc = torch.cat([pred_acc, pad], dim=0)
            else:
                pred_acc = pred_acc[:true_len]

            true_trim = true[:true_len]
            correct = pred_acc == true_trim
            is_exact = bool(correct.all().item())

            total_correct += int(correct.sum().item())
            total_tok += true_len
            total_seq += 1
            perfect += int(is_exact)

            pred_len_value = int(pred_len_i[i].item())
            length_error = pred_len_value - true_len

            pred_valid = preds[i]
            if pred_valid.numel() < pred_len_value:
                pad = torch.full(
                    (pred_len_value - pred_valid.numel(),),
                    PAD_ID,
                    device=pred_valid.device,
                    dtype=pred_valid.dtype,
                )
                pred_valid = torch.cat([pred_valid, pad], dim=0)
            else:
                pred_valid = pred_valid[:pred_len_value]

            pred_selfies = _ids_to_selfies(pred_valid.detach().cpu().tolist())
            is_valid = _reconstruction_is_valid(pred_selfies)

            true_len_rows.append(true_len)
            pred_len_rows.append(pred_len_value)
            length_error_rows.append(length_error)
            exact_match_rows.append(int(is_exact))
            valid_smiles_rows.append(int(is_valid))

    true_len_rows = np.asarray(true_len_rows, dtype=np.int32)
    pred_len_rows = np.asarray(pred_len_rows, dtype=np.int32)
    length_error_rows = np.asarray(length_error_rows, dtype=np.int32)
    exact_match_rows = np.asarray(exact_match_rows, dtype=np.int8)
    valid_smiles_rows = np.asarray(valid_smiles_rows, dtype=np.int8)

    per_sample_df = pd.DataFrame({
        "split": split_name,
        "true_len": true_len_rows,
        "pred_len": pred_len_rows,
        "length_error": length_error_rows,
        "is_exact_match": exact_match_rows,
        "is_valid_smiles": valid_smiles_rows,
    })

    summary = {
        "split": split_name,
        "token_accuracy": total_correct / max(total_tok, 1),
        "exact_sequence_accuracy": perfect / max(total_seq, 1),
        "length_mae": float(np.mean(np.abs(length_error_rows))),
        "length_rmse": float(np.sqrt(np.mean(np.square(length_error_rows.astype(np.float32))))),
        "exact_match_rate": float(np.mean(exact_match_rows)),
        "length_mismatch_rate": float(np.mean(pred_len_rows != true_len_rows)),
        "invalid_smiles_rate": float(1.0 - np.mean(valid_smiles_rows)),
    }

    return per_sample_df, summary


In [ ]:
split_data = {
    "train": train_data,
    "val": val_data,
    "test": test_data,
}

all_per_sample = []
phase1_summary_rows = []

for split_name, split_array in split_data.items():
    per_sample_df, split_summary = evaluate_phase1_split(split_name, split_array)
    all_per_sample.append(per_sample_df)
    phase1_summary_rows.append(split_summary)

    binned = per_sample_df.groupby("true_len", as_index=False)["length_error"].mean()
    plt.figure(figsize=(8, 5))
    plt.plot(binned["true_len"], binned["length_error"], marker="o", linewidth=1.5)
    plt.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
    plt.xlabel("True length (binned)")
    plt.ylabel("Mean length error (pred_len - true_len)")
    plt.title(f"Length error vs true length (binned) - {split_name}")
    plt.tight_layout()
    plt.savefig(PHASE1_DIR / f"phase1_{split_name}_length_error_vs_true_length_binned.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.hist(per_sample_df["length_error"], bins="auto")
    plt.xlabel("Length error (pred_len - true_len)")
    plt.ylabel("Count")
    plt.title(f"Length error histogram - {split_name}")
    plt.tight_layout()
    plt.savefig(PHASE1_DIR / f"phase1_{split_name}_length_error_histogram.png", dpi=150)
    plt.close()

phase1_per_sample_df = pd.concat(all_per_sample, ignore_index=True)
phase1_summary_df = pd.DataFrame(phase1_summary_rows)

phase1_per_sample_csv = PHASE1_DIR / "phase1_per_sample_metrics.csv"
phase1_summary_csv = PHASE1_DIR / "phase1_summary_by_split.csv"

phase1_per_sample_df.to_csv(phase1_per_sample_csv, index=False)
phase1_summary_df.to_csv(phase1_summary_csv, index=False)

for row in phase1_summary_rows:
    split = row["split"]
    metrics[f"phase1/{split}/token_accuracy"] = float(row["token_accuracy"])
    metrics[f"phase1/{split}/exact_sequence_accuracy"] = float(row["exact_sequence_accuracy"])
    metrics[f"phase1/{split}/length_mae"] = float(row["length_mae"])
    metrics[f"phase1/{split}/length_rmse"] = float(row["length_rmse"])
    metrics[f"phase1/{split}/exact_match_rate"] = float(row["exact_match_rate"])
    metrics[f"phase1/{split}/length_mismatch_rate"] = float(row["length_mismatch_rate"])
    metrics[f"phase1/{split}/invalid_smiles_rate"] = float(row["invalid_smiles_rate"])

print(f"Saved per-sample CSV: {phase1_per_sample_csv}")
print(f"Saved summary CSV: {phase1_summary_csv}")
phase1_summary_df


## Phase 2: Prior Sampling Benchmark
- Sample z ~ N(0, I) for N_GEN only when RUN_PHASE2 = True
- Decode z -> SELFIES -> SMILES -> RDKit
- Compute validity, uniqueness among valid, novelty vs TRAIN
- Save the same Phase 2 plots and tables only when the phase is enabled


In [ ]:
N_GEN = 5000

plots_dir = Path(OUTPUT_DIR) / 'plots'
tables_dir = Path(OUTPUT_DIR) / 'tables'
plots_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

if 'metrics' not in globals() or not isinstance(metrics, dict):
    metrics = {}

pad_id = PAD if isinstance(PAD, int) else (PAD_ID if 'PAD_ID' in globals() else 0)
sos_id = SOS if isinstance(SOS, int) else (SOS_ID if 'SOS_ID' in globals() else 1)
eos_id = EOS if isinstance(EOS, int) else (EOS_ID if 'EOS_ID' in globals() else 2)

def _extract_ids(item):
    if isinstance(item, (tuple, list)):
        item = item[0]
    if isinstance(item, torch.Tensor):
        item = item.detach().cpu().tolist()
    return [int(t) for t in item]

if RUN_PHASE2:
    train_canon_set = set()
    for item in tqdm(train_data, desc='Phase2 train canon set'):
        token_ids = _extract_ids(item)
        train_selfies = decode_tokens_to_selfies(token_ids, id2tok, pad_id, sos_id, eos_id)
        train_smiles = selfies_to_smiles(train_selfies)
        train_canon = canon_smiles(train_smiles)
        if train_canon is not None:
            train_canon_set.add(train_canon)

    latent_dim = int(FINAL_LATENT_SIZE)
    model.eval()
    with torch.no_grad():
        z = torch.randn(N_GEN, latent_dim, device=device)
        decoded = model.decode(z, mode='eval', sos_id=sos_id)

    logits = decoded[0] if isinstance(decoded, (tuple, list)) else decoded
    pred_len = decoded[1] if isinstance(decoded, (tuple, list)) and len(decoded) > 1 else None
    pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

    rows = []
    for i, ids in enumerate(tqdm(pred_ids, desc='Phase2 decode')):
        if pred_len is not None:
            L = max(1, int(torch.round(pred_len[i]).item()))
            ids = ids[:L] if len(ids) >= L else ids + [pad_id] * (L - len(ids))
        gen_selfies = decode_tokens_to_selfies(ids, id2tok, pad_id, sos_id, eos_id)
        gen_smiles = selfies_to_smiles(gen_selfies)
        gen_mol = smiles_to_mol(gen_smiles)
        is_valid = gen_mol is not None
        gen_canon = canon_smiles(gen_smiles) if is_valid else None
        rows.append({'sample_id': i, 'selfies': gen_selfies, 'smiles': gen_smiles, 'canon_smiles': gen_canon, 'is_valid': int(is_valid)})

    phase2_samples = pd.DataFrame(rows)
    valid_df = phase2_samples[phase2_samples['is_valid'] == 1].copy()
    valid_count = int(len(valid_df))
    unique_valid = set(valid_df['canon_smiles'].dropna().tolist())
    unique_valid_count = int(len(unique_valid))
    novel_unique_count = int(len(unique_valid - train_canon_set))
    validity = valid_count / float(N_GEN)
    uniqueness_among_valid = unique_valid_count / float(valid_count) if valid_count > 0 else 0.0
    novelty_vs_train = novel_unique_count / float(unique_valid_count) if unique_valid_count > 0 else 0.0

    plt.figure(figsize=(7, 4))
    metric_names = ['validity', 'uniqueness_among_valid', 'novelty_vs_train']
    metric_values = [validity, uniqueness_among_valid, novelty_vs_train]
    bars = plt.bar(metric_names, metric_values)
    plt.ylim(0, 1)
    for bar, val in zip(bars, metric_values):
        plt.text(bar.get_x() + bar.get_width() / 2.0, min(val + 0.02, 0.98), f'{val:.3f}', ha='center')
    plt.tight_layout()
    phase2_bar_path = plots_dir / 'phase2_metrics_bar.png'
    plt.savefig(phase2_bar_path, dpi=150)
    plt.close()

    plt.figure(figsize=(7, 4))
    valid_lengths = valid_df['canon_smiles'].dropna().str.len().to_numpy()
    plt.hist(valid_lengths, bins='auto')
    plt.xlabel('Canonical SMILES length')
    plt.ylabel('Count')
    plt.title('Phase 2: Valid canonical SMILES length histogram')
    plt.tight_layout()
    phase2_hist_path = plots_dir / 'phase2_valid_length_histogram.png'
    plt.savefig(phase2_hist_path, dpi=150)
    plt.close()

    phase2_head_csv = tables_dir / 'phase2_samples_head.csv'
    phase2_samples.head().to_csv(phase2_head_csv, index=False)
    metrics['phase2/validity'] = float(validity)
    metrics['phase2/uniqueness_among_valid'] = float(uniqueness_among_valid)
    metrics['phase2/novelty_vs_train'] = float(novelty_vs_train)
    metrics['phase2/status'] = 'ran'
    print(f'Saved bar plot: {phase2_bar_path}')
    print(f'Saved length histogram: {phase2_hist_path}')
    print(f'Saved samples head CSV: {phase2_head_csv}')
    phase2_result_df = pd.DataFrame([
        {'metric': 'validity', 'value': validity},
        {'metric': 'uniqueness_among_valid', 'value': uniqueness_among_valid},
        {'metric': 'novelty_vs_train', 'value': novelty_vs_train},
    ])
else:
    phase2_samples = None
    metrics['phase2/status'] = 'skipped'
    print('Phase 2 skipped intentionally because RUN_PHASE2 = False.')
    phase2_result_df = pd.DataFrame([{'metric': 'phase2_status', 'value': 'skipped'}])


In [ ]:
phase2_result_df


## Phase 3: Latent KL Diagnostics (Full Dataset)
- Use all splits combined (train + val + test)
- Reuse Step 3 latent cache when available (Z_mu.npy, Z_logvar.npy)
- Compute per-dimension KL mean (kl_dim) and total KL stats
- Compute active units count with fixed threshold
- Save KL plots, kl_dim CSV, and metrics entries


In [ ]:
from torch.utils.data import DataLoader

PHASE3_BATCH_SIZE = 1024
PHASE3_AU_THRESH = 0.01

phase3_plots_dir = Path(OUTPUT_DIR) / 'plots'
phase3_tables_dir = Path(OUTPUT_DIR) / 'tables'
phase3_plots_dir.mkdir(parents=True, exist_ok=True)
phase3_tables_dir.mkdir(parents=True, exist_ok=True)

if 'metrics' not in globals() or not isinstance(metrics, dict):
    metrics = {}

def _phase3_extract_tensor(item):
    if isinstance(item, (tuple, list)):
        item = item[0]
    if isinstance(item, np.ndarray):
        item = torch.from_numpy(item)
    return torch.as_tensor(item, dtype=torch.long)

all_items = []
for split in (train_data, val_data, test_data):
    for item in split:
        all_items.append(_phase3_extract_tensor(item))

n_expected = len(all_items)
step3_dir = Path(OUTPUT_DIR).resolve().parent / 'confounds_step3'
step3_z_mu_path = step3_dir / 'Z_mu.npy'
step3_z_logvar_path = step3_dir / 'Z_logvar.npy'
use_phase3_cache = False
phase3_source = 'fresh_encode'
phase3_reason = 'cache disabled'
kl_dim = None
total_kl = None

if REUSE_STEP3_LATENTS_FOR_PHASE3:
    if not step3_z_mu_path.exists():
        phase3_reason = f'missing cache file: {step3_z_mu_path}'
    elif not step3_z_logvar_path.exists():
        phase3_reason = f'missing cache file: {step3_z_logvar_path}'
    else:
        z_mu_cache = np.load(step3_z_mu_path)
        z_logvar_cache = np.load(step3_z_logvar_path)
        if z_mu_cache.ndim != 2 or z_logvar_cache.ndim != 2:
            phase3_reason = 'cache arrays are not rank-2'
        elif z_mu_cache.shape != z_logvar_cache.shape:
            phase3_reason = f'shape mismatch: mu={z_mu_cache.shape}, logvar={z_logvar_cache.shape}'
        elif z_mu_cache.shape[0] != n_expected:
            phase3_reason = f'row mismatch: cache={z_mu_cache.shape[0]}, expected={n_expected}'
        else:
            use_phase3_cache = True
            phase3_source = 'step3_cache'
            phase3_reason = 'using cached Step 3 latents'
            mu_all = torch.as_tensor(z_mu_cache, dtype=torch.float32)
            logvar_all = torch.as_tensor(z_logvar_cache, dtype=torch.float32)
            kl_all = 0.5 * (mu_all.pow(2) + logvar_all.exp() - 1.0 - logvar_all)
            kl_dim = kl_all.mean(dim=0).detach().cpu().numpy()
            total_kl = kl_all.sum(dim=1).detach().cpu().numpy().astype(np.float32)
            print(f'Phase 3: using Step 3 latent cache from {step3_dir}')

if not use_phase3_cache:
    if REUSE_STEP3_LATENTS_FOR_PHASE3:
        print(f'Phase 3: falling back to fresh encoding because {phase3_reason}.')
    loader = DataLoader(all_items, batch_size=PHASE3_BATCH_SIZE, shuffle=False)
    model.eval()
    kl_dim_sum = None
    total_kl_values = []
    n_samples = 0
    with torch.no_grad():
        for x in tqdm(loader, desc='Phase3 encode full dataset'):
            if isinstance(x, (tuple, list)):
                x = x[0]
            x = x.to(device)
            mu, logvar = model.encode(x)
            kl = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar)
            batch_kl_dim_sum = kl.sum(dim=0).detach().cpu()
            if kl_dim_sum is None:
                kl_dim_sum = batch_kl_dim_sum
            else:
                kl_dim_sum += batch_kl_dim_sum
            total_kl_values.extend(kl.sum(dim=1).detach().cpu().numpy().tolist())
            n_samples += int(x.size(0))
    kl_dim = (kl_dim_sum / max(n_samples, 1)).numpy()
    total_kl = np.asarray(total_kl_values, dtype=np.float32)

metrics['phase3/source'] = phase3_source
metrics['phase3/cache_message'] = phase3_reason


In [ ]:
total_kl_mean = float(np.mean(total_kl))
total_kl_median = float(np.median(total_kl))
total_kl_p10 = float(np.percentile(total_kl, 10))
total_kl_p90 = float(np.percentile(total_kl, 90))
active_units_count = int(np.sum(kl_dim > PHASE3_AU_THRESH))

kl_dim_df = pd.DataFrame({'dim': np.arange(len(kl_dim), dtype=np.int32), 'kl_dim': kl_dim})
phase3_kl_dim_csv = phase3_tables_dir / 'phase3_kl_dim.csv'
kl_dim_df.to_csv(phase3_kl_dim_csv, index=False)
kl_dim_sorted = np.sort(kl_dim)[::-1]

plt.figure(figsize=(8, 5))
plt.plot(np.arange(len(kl_dim_sorted)), kl_dim_sorted)
plt.xlabel('Latent dimension rank (sorted)')
plt.ylabel('Mean KL per dimension')
plt.title('Phase 3: kl_dim sorted')
plt.tight_layout()
phase3_kl_sorted_plot = phase3_plots_dir / 'phase3_kl_dim_sorted.png'
plt.savefig(phase3_kl_sorted_plot, dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.hist(kl_dim, bins='auto')
plt.xlabel('Mean KL per dimension')
plt.ylabel('Count')
plt.title('Phase 3: kl_dim histogram')
plt.tight_layout()
phase3_kl_hist_plot = phase3_plots_dir / 'phase3_kl_dim_histogram.png'
plt.savefig(phase3_kl_hist_plot, dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.hist(total_kl, bins='auto')
plt.xlabel('Total KL per sample')
plt.ylabel('Count')
plt.title('Phase 3: total KL histogram')
plt.tight_layout()
phase3_total_kl_hist_plot = phase3_plots_dir / 'phase3_total_kl_histogram.png'
plt.savefig(phase3_total_kl_hist_plot, dpi=150)
plt.close()

metrics['phase3/total_kl_mean'] = total_kl_mean
metrics['phase3/total_kl_median'] = total_kl_median
metrics['phase3/total_kl_p10'] = total_kl_p10
metrics['phase3/total_kl_p90'] = total_kl_p90
metrics['phase3/active_units_count'] = float(active_units_count)
metrics['phase3/active_units_thresh'] = float(PHASE3_AU_THRESH)
print(f'Saved kl_dim sorted plot: {phase3_kl_sorted_plot}')
print(f'Saved kl_dim histogram: {phase3_kl_hist_plot}')
print(f'Saved total KL histogram: {phase3_total_kl_hist_plot}')
print(f'Saved kl_dim CSV: {phase3_kl_dim_csv}')
pd.DataFrame([
    {'metric': 'total_kl_mean', 'value': total_kl_mean},
    {'metric': 'total_kl_median', 'value': total_kl_median},
    {'metric': 'total_kl_p10', 'value': total_kl_p10},
    {'metric': 'total_kl_p90', 'value': total_kl_p90},
    {'metric': 'active_units_count', 'value': active_units_count},
    {'metric': 'active_units_thresh', 'value': PHASE3_AU_THRESH},
    {'metric': 'source', 'value': phase3_source},
])


## Phase 4: Latent Interpolation (VAL Pairs)
- Sample random pairs from `val_data`
- Encode pair endpoints and interpolate `z(t)=(1-t)z1+t z2`
- Decode each step to `SELFIES -> SMILES -> RDKit` and compute validity
- Save required CSV/plots/artifacts and update Phase 4 metrics


In [ ]:
import json
from rdkit.Chem import Draw

N_PAIRS = 200
N_STEPS = 11
N_VIS = 5

phase4_plots_dir = Path(OUTPUT_DIR) / "plots"
phase4_tables_dir = Path(OUTPUT_DIR) / "tables"
phase4_plots_dir.mkdir(parents=True, exist_ok=True)
phase4_tables_dir.mkdir(parents=True, exist_ok=True)

if "metrics" not in globals() or not isinstance(metrics, dict):
    metrics = {}

pad_id = PAD if isinstance(PAD, int) else (PAD_ID if "PAD_ID" in globals() else 0)
sos_id = SOS if isinstance(SOS, int) else (SOS_ID if "SOS_ID" in globals() else 1)
eos_id = EOS if isinstance(EOS, int) else (EOS_ID if "EOS_ID" in globals() else 2)

def _phase4_extract_tensor(item):
    if isinstance(item, (tuple, list)):
        item = item[0]
    if isinstance(item, np.ndarray):
        item = torch.from_numpy(item)
    return torch.as_tensor(item, dtype=torch.long)

val_items = [_phase4_extract_tensor(item) for item in val_data]
n_val = len(val_items)
if n_val < 2:
    raise ValueError("Phase 4 requires at least two samples in val_data.")

rng = np.random.default_rng(SEED)
idx1 = rng.integers(0, n_val, size=N_PAIRS)
idx2 = rng.integers(0, n_val, size=N_PAIRS)
same = idx1 == idx2
while np.any(same):
    idx2[same] = rng.integers(0, n_val, size=int(np.sum(same)))
    same = idx1 == idx2

x1_batch = torch.stack([val_items[i] for i in idx1]).to(device)
x2_batch = torch.stack([val_items[i] for i in idx2]).to(device)

model.eval()
with torch.no_grad():
    z1_batch, _ = model.encode(x1_batch)
    z2_batch, _ = model.encode(x2_batch)

t_values = torch.linspace(0.0, 1.0, N_STEPS, device=device)
raw_rows = []

for pair_id in tqdm(range(N_PAIRS), desc="Phase4 interpolation"):
    z1 = z1_batch[pair_id]
    z2 = z2_batch[pair_id]
    z_interp = (1.0 - t_values).unsqueeze(1) * z1.unsqueeze(0) + t_values.unsqueeze(1) * z2.unsqueeze(0)

    with torch.no_grad():
        decoded = model.decode(z_interp, mode="eval", sos_id=sos_id)

    logits = decoded[0] if isinstance(decoded, (tuple, list)) else decoded
    pred_len = decoded[1] if isinstance(decoded, (tuple, list)) and len(decoded) > 1 else None
    pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

    for step, ids in enumerate(pred_ids):
        if pred_len is not None:
            L = max(1, int(torch.round(pred_len[step]).item()))
            ids = ids[:L] if len(ids) >= L else ids + [pad_id] * (L - len(ids))

        selfies_str = decode_tokens_to_selfies(ids, id2tok, pad_id, sos_id, eos_id)
        smiles_str = selfies_to_smiles(selfies_str)
        mol = smiles_to_mol(smiles_str)
        is_valid = int(mol is not None)
        canonical = canon_smiles(smiles_str) if is_valid else None

        raw_rows.append({
            "pair_id": int(pair_id),
            "step": int(step),
            "t": float(t_values[step].item()),
            "selfies": selfies_str,
            "smiles": smiles_str,
            "canonical_smiles": canonical,
            "valid": is_valid,
        })

phase4_raw_df = pd.DataFrame(raw_rows, columns=[
    "pair_id", "step", "t", "selfies", "smiles", "canonical_smiles", "valid"
])

phase4_raw_csv = phase4_tables_dir / "phase4_interp_raw.csv"
phase4_raw_df.to_csv(phase4_raw_csv, index=False)

validity_by_step = (
    phase4_raw_df.groupby("step", as_index=False)["valid"].mean()
    .sort_values("step")
    .reset_index(drop=True)
)

plt.figure(figsize=(8, 5))
plt.plot(validity_by_step["step"], validity_by_step["valid"], marker="o")
plt.ylim(0.0, 1.0)
plt.xlabel("Step")
plt.ylabel("Mean valid fraction")
plt.title("Interpolation validity by step")
plt.tight_layout()
phase4_validity_plot = phase4_plots_dir / "interp_validity_by_step.png"
plt.savefig(phase4_validity_plot, dpi=150)
plt.close()

vis_pair_ids = sorted(phase4_raw_df["pair_id"].unique().tolist())[: min(N_VIS, N_PAIRS)]
vis_rows = phase4_raw_df[phase4_raw_df["pair_id"].isin(vis_pair_ids)].copy()
vis_rows = vis_rows.sort_values(["pair_id", "step"]).reset_index(drop=True)

for pid in vis_pair_ids:
    one = vis_rows[vis_rows["pair_id"] == pid].sort_values("step")
    mols = [smiles_to_mol(s) for s in one["smiles"].tolist()]
    legends = [f"step={int(st)}" for st in one["step"].tolist()]
    img = Draw.MolsToGridImage(mols, molsPerRow=N_STEPS, subImgSize=(200, 200), legends=legends, useSVG=False, returnPNG=True)
    out_path = phase4_plots_dir / f"phase4_interp_pair_{pid}.png"
    if isinstance(img, (bytes, bytearray)):
        out_path.write_bytes(bytes(img))
    elif hasattr(img, "save"):
        img.save(str(out_path))
    elif hasattr(img, "data") and isinstance(img.data, (bytes, bytearray)):
        out_path.write_bytes(bytes(img.data))
    else:
        raise TypeError(f"Unsupported image type from MolsToGridImage: {type(img)}")

phase4_vis_smiles_csv = phase4_tables_dir / "phase4_interp_vis_smiles.csv"
vis_rows[["pair_id", "step", "t", "smiles", "canonical_smiles", "valid"]].to_csv(phase4_vis_smiles_csv, index=False)

overall_valid_fraction = float(phase4_raw_df["valid"].mean())
valid_fraction_by_step = validity_by_step["valid"].astype(float).tolist()

if "phase4" not in metrics or not isinstance(metrics.get("phase4"), dict):
    metrics["phase4"] = {}
metrics["phase4"]["overall_valid_fraction"] = overall_valid_fraction
metrics["phase4"]["valid_fraction_by_step"] = valid_fraction_by_step

metrics_json_path = Path(OUTPUT_DIR) / "metrics.json"
metrics_csv_path = Path(OUTPUT_DIR) / "metrics.csv"

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

csv_rows = []
for k, v in metrics.items():
    if isinstance(v, (dict, list)):
        value = json.dumps(v)
    else:
        value = v
    csv_rows.append({"metric": k, "value": value})
pd.DataFrame(csv_rows).to_csv(metrics_csv_path, index=False)

print(f"Saved raw CSV: {phase4_raw_csv}")
print(f"Saved validity plot: {phase4_validity_plot}")
print(f"Saved visual pair SMILES CSV: {phase4_vis_smiles_csv}")
print(f"Saved metrics JSON: {metrics_json_path}")
print(f"Saved metrics CSV: {metrics_csv_path}")

metrics["phase4"]


## Phase 4b: Interpolation Continuity Cleanup (`_v2`)

This pass keeps the original phase-4 interpolation as the baseline and adds a second, denser trajectory with local decode packs.

The goals are:

- keep the original CSV evidence intact
- use a center-dense step schedule
- generate a small local decode pack per step using orthogonal latent noise
- save a medoid valid representative per step
- compare baseline versus v2 with adjacent-step fingerprint continuity


In [ ]:
from pathlib import Path
from rdkit import DataStructs
from rdkit.Chem import AllChem

STEP4_DIR = Path(OUTPUT_DIR).resolve().parent / "directions_step4"
STEP4_SCALERS = STEP4_DIR / "scalers.npz"
assert STEP4_SCALERS.exists(), f"Missing Step 4 scalers: {STEP4_SCALERS}"

scalers_npz = np.load(STEP4_SCALERS, allow_pickle=True)
z_std = scalers_npz["z_std"].astype(np.float32)
SIGMA_LOW = 0.05 * float(np.median(z_std))
SIGMA_HIGH = 0.10 * float(np.median(z_std))
PHASE4_V2_T_VALUES = np.array([
    0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
    0.48, 0.50, 0.52, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.90, 1.00
], dtype=np.float32)
N_STEP_SAMPLES_V2 = 5
N_PAIRS_V2 = 200
N_VIS_V2 = 5
phase4_raw_v2_csv = phase4_tables_dir / "phase4_interp_raw_v2.csv"
phase4_samples_v2_csv = phase4_tables_dir / "phase4_interp_samples_v2.csv"
phase4_vis_v2_csv = phase4_tables_dir / "phase4_interp_vis_smiles_v2.csv"
phase4_continuity_plot_v2 = phase4_plots_dir / "phase4_interp_continuity_comparison_v2.png"


In [ ]:
def _phase4_v2_decode_latents(z_batch):
    with torch.no_grad():
        decoded = model.decode(z_batch, mode="eval", sos_id=SOS_ID)
    logits = decoded[0] if isinstance(decoded, (tuple, list)) else decoded
    pred_len = decoded[1] if isinstance(decoded, (tuple, list)) and len(decoded) > 1 else None
    pred_ids = logits.argmax(dim=-1)

    rows = []
    for i in range(pred_ids.size(0)):
        ids = pred_ids[i]
        pred_len_value = None
        if pred_len is not None:
            pred_len_value = max(1, int(torch.round(pred_len[i]).item()))
            ids = ids[:pred_len_value]
        ids_list = ids.detach().cpu().tolist()
        selfies_str = decode_tokens_to_selfies(ids_list, id2tok, PAD_ID, SOS_ID, EOS_ID)
        smiles_str = selfies_to_smiles(selfies_str)
        mol = smiles_to_mol(smiles_str)
        canon = canon_smiles(smiles_str) if mol is not None else None
        rows.append({
            "selfies": selfies_str,
            "smiles": smiles_str,
            "canonical_smiles": canon,
            "valid": int(canon is not None),
            "pred_len": pred_len_value if pred_len_value is not None else len(ids_list),
        })
    return pd.DataFrame(rows)

def _orthogonal_noise(base_direction, sigma, n_samples, rng_seed_offset):
    direction = base_direction.detach().cpu().numpy().astype(np.float32)
    direction_norm = np.linalg.norm(direction)
    if direction_norm < 1e-8 or sigma <= 0:
        return np.zeros((n_samples, direction.shape[0]), dtype=np.float32)
    unit = direction / direction_norm
    rng_local = np.random.default_rng(SEED + int(rng_seed_offset))
    noise = rng_local.normal(size=(n_samples, direction.shape[0])).astype(np.float32)
    proj = (noise @ unit)[:, None] * unit[None, :]
    noise = noise - proj
    norms = np.linalg.norm(noise, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-8, None)
    return sigma * noise / norms

def _make_fp(smiles_str):
    mol = smiles_to_mol(smiles_str)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

def _medoid_valid_row(step_df):
    valid = step_df[step_df["valid"] == 1].copy()
    if valid.empty:
        row = step_df.iloc[0].copy()
        row["medoid_valid_count"] = 0
        row["medoid_unique_valid_count"] = 0
        return row
    if len(valid) == 1:
        row = valid.iloc[0].copy()
        row["medoid_valid_count"] = 1
        row["medoid_unique_valid_count"] = 1
        return row

    fps = [_make_fp(smi) for smi in valid["canonical_smiles"]]
    dist_sums = []
    for i, fp_i in enumerate(fps):
        total = 0.0
        for j, fp_j in enumerate(fps):
            if i == j:
                continue
            total += 1.0 - DataStructs.TanimotoSimilarity(fp_i, fp_j)
        dist_sums.append(total)
    medoid_idx = int(np.argmin(dist_sums))
    row = valid.iloc[medoid_idx].copy()
    row["medoid_valid_count"] = int(len(valid))
    row["medoid_unique_valid_count"] = int(valid["canonical_smiles"].nunique())
    return row

def _adjacent_similarity_from_rows(df_in, label):
    rows = []
    for pair_id, pair_df in df_in.groupby("pair_id"):
        pair_df = pair_df.sort_values("step").reset_index(drop=True)
        prev_fp = None
        prev_t = None
        prev_valid = None
        for row in pair_df.itertuples():
            fp = _make_fp(row.canonical_smiles) if int(row.valid) == 1 else None
            if prev_fp is not None and fp is not None and prev_valid == 1 and int(row.valid) == 1:
                rows.append({
                    "variant": label,
                    "pair_id": int(pair_id),
                    "step_prev": int(row.step - 1),
                    "step_curr": int(row.step),
                    "t_mid": 0.5 * (float(prev_t) + float(row.t)),
                    "adjacent_tanimoto": float(DataStructs.TanimotoSimilarity(prev_fp, fp)),
                })
            prev_fp = fp
            prev_t = float(row.t)
            prev_valid = int(row.valid)
    return pd.DataFrame(rows)


In [ ]:
model.eval()
rng_phase4_v2 = np.random.default_rng(SEED)
idx1_v2 = rng_phase4_v2.integers(0, len(val_data), size=N_PAIRS_V2)
idx2_v2 = rng_phase4_v2.integers(0, len(val_data), size=N_PAIRS_V2)
same_mask = idx1_v2 == idx2_v2
while np.any(same_mask):
    idx2_v2[same_mask] = rng_phase4_v2.integers(0, len(val_data), size=int(np.sum(same_mask)))
    same_mask = idx1_v2 == idx2_v2

x1_v2 = torch.stack([_phase4_extract_tensor(val_data[i]) for i in idx1_v2]).to(device)
x2_v2 = torch.stack([_phase4_extract_tensor(val_data[i]) for i in idx2_v2]).to(device)

with torch.no_grad():
    z1_v2, _ = model.encode(x1_v2)
    z2_v2, _ = model.encode(x2_v2)

rep_rows = []
sample_rows = []
sigma_current = SIGMA_LOW
pack_uniqueness_scores = []
sigma_upgraded = False

for pair_id in tqdm(range(N_PAIRS_V2), desc="Phase4 interpolation v2"):
    z1 = z1_v2[pair_id]
    z2 = z2_v2[pair_id]
    direction = z2 - z1
    t_tensor = torch.tensor(PHASE4_V2_T_VALUES, dtype=torch.float32, device=device)
    z_interp = (1.0 - t_tensor).unsqueeze(1) * z1.unsqueeze(0) + t_tensor.unsqueeze(1) * z2.unsqueeze(0)

    local_rows = []
    for step_idx, t_value in enumerate(PHASE4_V2_T_VALUES):
        z_center = z_interp[step_idx]
        noise = _orthogonal_noise(direction, sigma_current, N_STEP_SAMPLES_V2, pair_id * 1000 + step_idx)
        z_pack = z_center.unsqueeze(0) + torch.tensor(noise, dtype=torch.float32, device=device)
        decoded_pack = _phase4_v2_decode_latents(z_pack)
        decoded_pack["pair_id"] = pair_id
        decoded_pack["step"] = step_idx
        decoded_pack["t"] = float(t_value)
        decoded_pack["sample_idx"] = np.arange(len(decoded_pack), dtype=int)
        sample_rows.extend(decoded_pack.to_dict("records"))
        rep = _medoid_valid_row(decoded_pack)
        rep_rows.append({
            "pair_id": pair_id,
            "step": step_idx,
            "t": float(t_value),
            "selfies": rep["selfies"],
            "smiles": rep["smiles"],
            "canonical_smiles": rep["canonical_smiles"],
            "valid": int(rep["valid"]),
            "medoid_valid_count": int(rep["medoid_valid_count"]),
            "medoid_unique_valid_count": int(rep["medoid_unique_valid_count"]),
            "sigma_used": float(sigma_current),
        })
        local_rows.append(rep["medoid_unique_valid_count"])

    if pair_id < 3:
        pack_uniqueness_scores.append(float(np.mean(local_rows)))
        if (pair_id == 2) and (not sigma_upgraded):
            if float(np.mean(pack_uniqueness_scores)) < 1.05:
                sigma_current = SIGMA_HIGH
                sigma_upgraded = True

phase4_rep_v2_df = pd.DataFrame(rep_rows)
phase4_samples_v2_df = pd.DataFrame(sample_rows)
phase4_rep_v2_df.to_csv(phase4_raw_v2_csv, index=False)
phase4_samples_v2_df.to_csv(phase4_samples_v2_csv, index=False)

vis_pair_ids_v2 = sorted(phase4_rep_v2_df["pair_id"].unique().tolist())[: min(N_VIS_V2, N_PAIRS_V2)]
phase4_rep_v2_df[phase4_rep_v2_df["pair_id"].isin(vis_pair_ids_v2)][
    ["pair_id", "step", "t", "smiles", "canonical_smiles", "valid", "medoid_valid_count", "medoid_unique_valid_count"]
].to_csv(phase4_vis_v2_csv, index=False)

baseline_path = phase4_tables_dir / "phase4_interp_raw.csv"
baseline_df = pd.read_csv(baseline_path)
baseline_adj = _adjacent_similarity_from_rows(baseline_df, "baseline")
v2_adj = _adjacent_similarity_from_rows(phase4_rep_v2_df, "v2")
adj_df = pd.concat([baseline_adj, v2_adj], ignore_index=True)

plot_df = adj_df.groupby(["variant", "t_mid"], as_index=False)["adjacent_tanimoto"].median()
plt.figure(figsize=(9, 5))
for variant, variant_df in plot_df.groupby("variant"):
    plt.plot(variant_df["t_mid"], variant_df["adjacent_tanimoto"], marker="o", linewidth=1.8, label=variant)
plt.axvspan(0.40, 0.60, color="grey", alpha=0.15, label="center window")
plt.xlabel("Midpoint t of adjacent step pair")
plt.ylabel("Median adjacent-step Tanimoto")
plt.title("Phase 4 continuity: baseline vs center-dense v2")
plt.legend()
plt.tight_layout()
plt.savefig(phase4_continuity_plot_v2, dpi=160)
plt.show()

center_baseline = baseline_adj[(baseline_adj["t_mid"] >= 0.40) & (baseline_adj["t_mid"] <= 0.60)]["adjacent_tanimoto"].median()
center_v2 = v2_adj[(v2_adj["t_mid"] >= 0.40) & (v2_adj["t_mid"] <= 0.60)]["adjacent_tanimoto"].median()
print("sigma low:", SIGMA_LOW)
print("sigma high:", SIGMA_HIGH)
print("sigma upgraded:", sigma_upgraded)
print("center median adjacent tanimoto baseline:", center_baseline)
print("center median adjacent tanimoto v2:", center_v2)


In [ ]:
from rdkit.Chem import Draw

phase4_v2_image_rows = []
for pid in vis_pair_ids_v2:
    one = phase4_rep_v2_df[phase4_rep_v2_df["pair_id"] == pid].sort_values("step").reset_index(drop=True)
    mols = [smiles_to_mol(s) for s in one["smiles"].tolist()]
    legends = [
        f"step={int(row.step)}\nt={float(row.t):.2f}\nvalid={int(row.valid)}"
        for row in one.itertuples()
    ]
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=len(one),
        subImgSize=(200, 200),
        legends=legends,
        useSVG=False,
        returnPNG=True,
    )
    out_path = phase4_plots_dir / f"phase4_interp_pair_{pid}_v2.png"
    if isinstance(img, (bytes, bytearray)):
        out_path.write_bytes(bytes(img))
    elif hasattr(img, "save"):
        img.save(str(out_path))
    elif hasattr(img, "data") and isinstance(img.data, (bytes, bytearray)):
        out_path.write_bytes(bytes(img.data))
    else:
        raise TypeError(f"Unsupported image type from MolsToGridImage: {type(img)}")
    phase4_v2_image_rows.append({"pair_id": int(pid), "path": str(out_path)})

phase4_v2_image_df = pd.DataFrame(phase4_v2_image_rows)
print("Saved v2 traversal images:")
phase4_v2_image_df


## Phase 5: Direct Same-Family Traversals
- Use the original dataset split already built in this notebook
- Detect functional groups directly from validation SMILES
- Sample one random endpoint pair per family
- Interpolate between the two molecules and record the decoded path


In [ ]:
import json

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, rdFingerprintGenerator
from sklearn.model_selection import train_test_split

PHASE5_TARGET_FAMILIES = None  # None means run all eligible families
PHASE5_RANDOM_SEED = 42
PHASE5_MIN_GROUP_SIZE = 20
PHASE5_T_VALUES = np.asarray(PHASE4_V2_T_VALUES, dtype=float) if 'PHASE4_V2_T_VALUES' in globals() else np.linspace(0.0, 1.0, 11)

phase5_tables_dir = Path(OUTPUT_DIR) / 'tables'
phase5_plots_dir = Path(OUTPUT_DIR) / 'plots'
phase5_family_images_dir = phase5_plots_dir / 'phase5_family_traversals'
phase5_tables_dir.mkdir(parents=True, exist_ok=True)
phase5_plots_dir.mkdir(parents=True, exist_ok=True)
phase5_family_images_dir.mkdir(parents=True, exist_ok=True)

phase5_group_membership_csv = phase5_tables_dir / 'phase5_functional_group_membership.csv'
phase5_traversal_csv = phase5_tables_dir / 'phase5_family_traversal_records.csv'
phase5_all_tanimoto_plot = phase5_plots_dir / 'phase5_all_family_tanimoto.png'

if 'metrics' not in globals() or not isinstance(metrics, dict):
    metrics = {}

for key in [
    'phase5/overall_valid_fraction',
    'phase5/overall_target_family_retention',
    'phase5/pair_counts_by_target_family',
]:
    metrics.pop(key, None)

if RUN_PHASE5_SEMANTIC_INTERP:
    if 'df' not in globals() or 'padded_data' not in globals() or 'val_data' not in globals():
        raise ValueError('Phase 5 requires the original dataset setup cells to run first.')

    dataset_indices = np.arange(len(padded_data))
    train_idx, temp_idx = train_test_split(dataset_indices, test_size=0.2, random_state=42, shuffle=True)
    val_idx, _ = train_test_split(temp_idx, test_size=0.5, random_state=42, shuffle=True)

    val_meta = df.iloc[val_idx].reset_index(drop=True).copy()
    val_meta['val_pos'] = np.arange(len(val_meta), dtype=int)
    val_tokens = np.asarray(val_data)
    expected_val_tokens = np.asarray(padded_data[val_idx])
    if val_tokens.shape != expected_val_tokens.shape or not np.array_equal(val_tokens, expected_val_tokens):
        raise ValueError('Phase 5 could not align validation metadata with val_data.')

    phase5_functional_group_smarts = [
        ('alcohol', '[OX2H][CX4]'),
        ('phenol', '[OX2H][c]'),
        ('ether', '[OD2]([#6;!$(C=O)])[#6;!$(C=O)]'),
        ('amine', '[NX3;H2,H1,H0;!$(N[C,S,P]=O);!$(N=*);!$(N#*)]'),
        ('amide', '[NX3][CX3](=[OX1])[#6,#7,#8]'),
        ('carboxylic_acid', '[CX3](=O)[OX2H1]'),
        ('ester', '[CX3](=O)[OX2H0][#6]'),
        ('aldehyde', '[CX3H1](=O)[#6]'),
        ('ketone', '[#6][CX3](=O)[#6]'),
        ('nitrile', '[CX2]#N'),
        ('sulfonamide', '[SX4](=[OX1])(=[OX1])([#6])[NX3]'),
    ]
    phase5_patterns = []
    for family_name, smarts in phase5_functional_group_smarts:
        patt = Chem.MolFromSmarts(smarts)
        if patt is None:
            raise ValueError(f'Invalid SMARTS for {family_name}: {smarts}')
        phase5_patterns.append((family_name, patt))

    phase5_fp_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

    def _phase5_detect_target_families(mol):
        if mol is None:
            return []
        try:
            return [family_name for family_name, patt in phase5_patterns if mol.HasSubstructMatch(patt)]
        except Exception:
            return []

    def _phase5_fp_from_mol(mol):
        if mol is None:
            return None
        return phase5_fp_generator.GetFingerprint(mol)

    def _phase5_similarity(fp1, fp2):
        if fp1 is None or fp2 is None:
            return np.nan
        return float(DataStructs.TanimotoSimilarity(fp1, fp2))

    val_meta['canonical_smiles'] = val_meta['smiles'].apply(canon_smiles)
    val_meta['target_families'] = val_meta['smiles'].apply(lambda s: _phase5_detect_target_families(smiles_to_mol(s)))
    val_meta['target_families_str'] = val_meta['target_families'].apply(lambda fams: ';'.join(fams))
    val_meta = val_meta[val_meta['canonical_smiles'].notna()].copy()
    phase5_membership_df = val_meta.explode('target_families').rename(columns={'target_families': 'target_family'}).reset_index(drop=True)
    phase5_membership_df = phase5_membership_df.dropna(subset=['target_family']).copy()
    phase5_membership_df['target_family'] = phase5_membership_df['target_family'].astype(str)
    phase5_membership_df.to_csv(phase5_group_membership_csv, index=False)

    available_counts = phase5_membership_df['target_family'].value_counts().sort_values(ascending=False)
    eligible_families = [family for family, count in available_counts.items() if int(count) >= PHASE5_MIN_GROUP_SIZE]
    if PHASE5_TARGET_FAMILIES is None:
        selected_families = eligible_families
    else:
        selected_families = [family for family in PHASE5_TARGET_FAMILIES if family in eligible_families]

    if not selected_families:
        available = ', '.join(available_counts.index.tolist())
        raise ValueError(f'No eligible Phase 5 families selected. Available groups: {available}')

    rng_phase5 = np.random.default_rng(PHASE5_RANDOM_SEED)
    traversal_rows = []
    tanimoto_rows = []
    successful_families = []

    for pair_id, target_family in enumerate(selected_families):
        family_df = (
            phase5_membership_df[phase5_membership_df['target_family'] == target_family]
            .drop_duplicates(subset=['canonical_smiles'])
            .reset_index(drop=True)
            .copy()
        )
        if len(family_df) < 2:
            print(f"Phase 5 skip: '{target_family}' does not have two distinct molecules.")
            continue

        chosen_positions = rng_phase5.choice(len(family_df), size=2, replace=False)
        start_row = family_df.iloc[int(chosen_positions[0])]
        end_row = family_df.iloc[int(chosen_positions[1])]

        start_tensor = _phase4_extract_tensor(val_data[int(start_row.val_pos)]).unsqueeze(0).to(device)
        end_tensor = _phase4_extract_tensor(val_data[int(end_row.val_pos)]).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            z1, _ = model.encode(start_tensor)
            z2, _ = model.encode(end_tensor)
        z1 = z1[0]
        z2 = z2[0]

        start_fp = _phase5_fp_from_mol(smiles_to_mol(start_row.smiles))
        end_fp = _phase5_fp_from_mol(smiles_to_mol(end_row.smiles))

        print('')
        print(f'Phase 5 family: {target_family}')
        print(f'  start: {start_row.smiles}')
        print(f'  end:   {end_row.smiles}')

        family_rows = []
        prev_fp = None
        prev_step = None
        prev_t = None
        for step_idx, t_value in enumerate(PHASE5_T_VALUES):
            t_scalar = float(t_value)
            z = (1.0 - t_scalar) * z1 + t_scalar * z2
            decoded = _phase4_v2_decode_latents(z.unsqueeze(0))
            decoded_row = decoded.iloc[0]
            mol = smiles_to_mol(decoded_row.smiles)
            fp = _phase5_fp_from_mol(mol)
            matched_families = _phase5_detect_target_families(mol)
            matched_label = ';'.join(matched_families)
            row_record = {
                'pair_id': int(pair_id),
                'target_family': target_family,
                'start_smiles': start_row.smiles,
                'end_smiles': end_row.smiles,
                'step': int(step_idx),
                't': t_scalar,
                'selfies': decoded_row.selfies,
                'smiles': decoded_row.smiles,
                'canonical_smiles': decoded_row.canonical_smiles,
                'valid': int(decoded_row.valid),
                'retention': int(target_family in matched_families),
                'matched_families': matched_label,
                'similarity_to_start': _phase5_similarity(fp, start_fp),
                'similarity_to_end': _phase5_similarity(fp, end_fp),
            }
            family_rows.append(row_record)
            traversal_rows.append(row_record)
            print(
                f"  step={step_idx:02d} "
                f"t={t_scalar:.2f} "
                f"valid={int(row_record['valid'])} "
                f"retained={int(row_record['retention'])} "
                f"family={matched_label if matched_label else '-'} "
                f"smiles={row_record['smiles']}"
            )

            if prev_fp is not None and fp is not None:
                tanimoto_rows.append({
                    'target_family': target_family,
                    'step_prev': int(prev_step),
                    'step_curr': int(step_idx),
                    't_mid': 0.5 * (float(prev_t) + t_scalar),
                    'adjacent_step_tanimoto': float(DataStructs.TanimotoSimilarity(prev_fp, fp)),
                })
            prev_fp = fp
            prev_step = int(step_idx)
            prev_t = t_scalar

        family_traversal_df = pd.DataFrame(family_rows)
        mols = [smiles_to_mol(s) for s in family_traversal_df['smiles'].tolist()]
        legends = [
            f"{row.target_family}\nstep={int(row.step)} t={float(row.t):.2f}\nvalid={int(row.valid)} retained={int(row.retention)}"
            for row in family_traversal_df.itertuples(index=False)
        ]
        img = Draw.MolsToGridImage(
            mols,
            molsPerRow=len(mols),
            subImgSize=(320, 320),
            legends=legends,
            useSVG=False,
            returnPNG=True,
        )
        image_path = phase5_family_images_dir / f'phase5_traversal_{target_family}.png'
        if isinstance(img, (bytes, bytearray)):
            image_path.write_bytes(bytes(img))
        elif hasattr(img, 'save'):
            img.save(str(image_path))
        elif hasattr(img, 'data') and isinstance(img.data, (bytes, bytearray)):
            image_path.write_bytes(bytes(img.data))
        else:
            raise TypeError(f'Unsupported image type from MolsToGridImage: {type(img)}')

        successful_families.append(target_family)

    if not traversal_rows:
        raise ValueError('Phase 5 did not produce any family traversals.')

    phase5_traversal_df = pd.DataFrame(traversal_rows)
    phase5_traversal_df.to_csv(phase5_traversal_csv, index=False)
    phase5_tanimoto_df = pd.DataFrame(tanimoto_rows)

    if not phase5_tanimoto_df.empty:
        plot_df = phase5_tanimoto_df.groupby(['target_family', 't_mid'], as_index=False)['adjacent_step_tanimoto'].median()
        plt.figure(figsize=(10, 6))
        for target_family, family_plot_df in plot_df.groupby('target_family'):
            plt.plot(
                family_plot_df['t_mid'],
                family_plot_df['adjacent_step_tanimoto'],
                marker='o',
                linewidth=1.8,
                label=target_family,
            )
        plt.axvspan(0.40, 0.60, color='grey', alpha=0.15, label='center window')
        plt.xlabel('Midpoint t of adjacent steps')
        plt.ylabel('Adjacent-step Tanimoto')
        plt.title('Phase 5 continuity across direct same-family traversals')
        plt.legend()
        plt.tight_layout()
        plt.savefig(phase5_all_tanimoto_plot, dpi=180)
        plt.close()
    else:
        plt.figure(figsize=(8, 5))
        plt.text(0.5, 0.5, 'No valid adjacent-step pairs', ha='center', va='center')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(phase5_all_tanimoto_plot, dpi=180)
        plt.close()

    metrics['phase5/overall_valid_fraction'] = float(phase5_traversal_df['valid'].mean())
    metrics['phase5/overall_target_family_retention'] = float(phase5_traversal_df['retention'].mean())
    metrics['phase5/pair_counts_by_target_family'] = {family: 1 for family in successful_families}

    metrics_json_path = Path(OUTPUT_DIR) / 'metrics.json'
    metrics_csv_path = Path(OUTPUT_DIR) / 'metrics.csv'
    with open(metrics_json_path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2)

    csv_rows = []
    for key, value in metrics.items():
        csv_rows.append({
            'metric': key,
            'value': json.dumps(value) if isinstance(value, (dict, list)) else value,
        })
    pd.DataFrame(csv_rows).to_csv(metrics_csv_path, index=False)

    print('')
    print(f'Saved functional-group membership table: {phase5_group_membership_csv}')
    print(f'Saved traversal record table: {phase5_traversal_csv}')
    print(f'Saved combined Tanimoto plot: {phase5_all_tanimoto_plot}')
    print(f'Saved family traversal images in: {phase5_family_images_dir}')
else:
    print('Phase 5 skipped because RUN_PHASE5_SEMANTIC_INTERP = False.')
